In [1]:
import boto3
import json
import os
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

True

In [25]:
df = pd.read_csv('../data/news.csv')
df = df[df["Text"].str.contains("terrorism", case=False)]
df

,Text,terrorism,security,espionage,communalism
38,The Monetary Authority of Singapore (MAS) has ...,True,False,False,False
119,SINGAPORE: Southeast Asia need not worry about...,True,False,False,False
833,Singapore and the Philippines are looking to b...,True,False,False,False
936,"Kevin Francis Hawkins, 30, was suffering from ...",True,False,False,False
1223,"""China offered to support long-time strategic ...",True,True,False,False
1367,The U.S. drone killing of American-born and -r...,True,False,False,False
1492,The Monetary Authority of Singapore (MAS) has ...,True,False,False,False


In [26]:
# Define the template for Bedrock request
def generate_bedrock_request(query: str) -> str:
    return (json.dumps
        ({
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 1000,
        "system": "You are an assistant helping Singapore's Internal Security Department (ISD) in addressing"
                          "four critical areas of concern: Espionage, Terrorism and Violent Extremism, Cybersecurity,"
                          "and Communalism. Using datasets from WikiLeaks articles and news excerpts, the goal is to"
                          "extract structured data to identify key entities, relationships, and incidents relevant to"
                          "Singapore’s internal security. The data may include unrelated global events (e.g., the"
                          "Taliban takeover in Afghanistan), but these are still valuable for identifying patterns,"
                          "trends, and lessons that could impact Singapore indirectly. By scoring relevance and"
                          "extracting actionable insights, ISD can prioritize risks, allocate resources effectively,"
                          "and maintain a proactive stance against both domestic and international security threats.",
        "messages": [
            {
                "role": "user",
                "content":
                    f"Provide the output of the text in the format: "
                    f"- Entity "
                    f"- Type (Person, Organization, Location, Incident, Keyword) "
                    f"- Date (if mentioned) "
                    f"- Summary (brief overview of relevance to ISD's concerns)"
                    f"- Area of Concern (Espionage, Terrorism, Cybersecurity, Communalism)"
                    f"- Relevance Score (High, Medium, Low)"
                    f"- Relationships (if applicable, e.g., Person A linked to Organization B)."
                    f"Here is the text:"
                    f"{query}"
            }
        ]
    }
    ))

def generate_bedrock_request_ch(query: str) -> str:
    return (json.dumps
        ({
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 1000,
        "system": "You are an assistant helping Singapore's Internal Security Department (ISD) in addressing four critical areas of concern: Espionage, Terrorism and Violent Extremism, Cybersecurity, and Communalism. ",
        "messages": [
            {
                "role": "user",
                "content":
                    f'Given a text taken from a news source, identify if the news is relevant to the four critical areas of concern of ISD, and extract structured data to identify key entities, relationships, and incidents relevant to Singapore’s internal security.'
                    f'Provide the output following the example JSON format:'
                    '{'
                    'Entities and Types: [[USA, Country], [Meta, Organisation]],'
                    'Date: 2025-01-30 or NA,'
                    'Summary: Terrorism attack in the USA,'
                    'Area of Concern: Terrorism,'
                    'Relevance Score: High,'
                    'Relationships: Meta USA attacked by terrorist group'
                    '}'
                f'Here is the text:'
                    f'{query}'
            }
        ]
    }
    ))

In [28]:
json.loads(generate_bedrock_request(df.iloc[0]['Text']))

{'anthropic_version': 'bedrock-2023-05-31',
 'max_tokens': 1000,
 'system': "You are an assistant helping Singapore's Internal Security Department (ISD) in addressingfour critical areas of concern: Espionage, Terrorism and Violent Extremism, Cybersecurity,and Communalism. Using datasets from WikiLeaks articles and news excerpts, the goal is toextract structured data to identify key entities, relationships, and incidents relevant toSingapore’s internal security. The data may include unrelated global events (e.g., theTaliban takeover in Afghanistan), but these are still valuable for identifying patterns,trends, and lessons that could impact Singapore indirectly. By scoring relevance andextracting actionable insights, ISD can prioritize risks, allocate resources effectively,and maintain a proactive stance against both domestic and international security threats.",
 'messages': [{'role': 'user',
   'content': 'Provide the output of the text in the format: - Entity - Type (Person, Organizat

In [23]:
client = boto3.client(
    'bedrock-runtime',
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    region_name='ap-southeast-1'
)
response = client.invoke_model(
    body=generate_bedrock_request(df.iloc[0]['text']),
    modelId='anthropic.claude-3-haiku-20240307-v1:0',
    contentType='application/json',
    accept='application/json'
)

In [24]:
answer = json.loads(response.get('body').read())

In [28]:
print(answer['content'][0]['Text'])

- Entity: Monetary Authority of Singapore (MAS)
  - Type: Organization
  - Date: N/A
  - Summary: The MAS fined several banks and an insurer for breaching requirements on anti-money laundering and countering terrorism financing. This indicates the MAS's role in monitoring and enforcing compliance with regulations related to financial crimes.
  - Area of Concern: Terrorism, Cybersecurity
  - Relevance Score: High
  - Relationships: MAS fined DBS, OCBC, Citibank, and Swiss Life.

- Entity: DBS
- Type: Organization
- Date: N/A
- Summary: DBS was fined S$2.6 million by the MAS for inadequate money laundering and terrorism financing controls when dealing with individuals and entities linked to Wirecard AG.
- Area of Concern: Terrorism, Cybersecurity
- Relevance Score: High
- Relationships: Fined by MAS

- Entity: OCBC
- Type: Organization
- Date: N/A
- Summary: OCBC was fined S$600,000 by the MAS for inadequate money laundering and terrorism financing controls when dealing with individuals 

In [32]:
json.loads(generate_bedrock_request_ch(df.iloc[0]['Text']))

{'anthropic_version': 'bedrock-2023-05-31',
 'max_tokens': 1000,
 'system': "You are an assistant helping Singapore's Internal Security Department (ISD) in addressing four critical areas of concern: Espionage, Terrorism and Violent Extremism, Cybersecurity, and Communalism. ",
 'messages': [{'role': 'user',
   'content': 'Given a text taken from a news source, identify if the news is relevant to the four critical areas of concern of ISD, and extract structured data to identify key entities, relationships, and incidents relevant to Singapore’s internal security.Provide the output following the example JSON format:{Entities and Types: [[USA, Country], [Meta, Organisation]],Date: 2025-01-30 or NA,Summary: Terrorism attack in the USA,Area of Concern: Terrorism,Relevance Score: High,Relationships: Meta USA attacked by terrorist group}Here is the text:The Monetary Authority of Singapore (MAS) has fined banks DBS, OCBC, Citibank and insurer Swiss Life a total of S$3.8 million (US$2.83 million

In [39]:
client = boto3.client(
    'bedrock-runtime',
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    region_name='ap-southeast-1'
)
response = client.invoke_model(
    body=generate_bedrock_request_ch(df.iloc[0]['text']),
    modelId='anthropic.claude-3-haiku-20240307-v1:0',
    contentType='application/json',
    accept='application/json'
)

In [41]:
answer2 = json.loads(response.get('body').read())

In [45]:
print(answer2['content'][0]['Text'])

{
"Entities and Types": [
    ["Monetary Authority of Singapore", "Organisation"],
    ["DBS", "Organisation"],
    ["OCBC", "Organisation"],
    ["Citibank", "Organisation"],
    ["Swiss Life", "Organisation"],
    ["Wirecard AG", "Organisation"]
],
"Date": "2022-12-16",
"Summary": "The Monetary Authority of Singapore has fined four banks and an insurer a total of S$3.8 million for breaching its requirements on anti-money laundering and countering terrorism financing. The breaches were identified when MAS examined the financial institutions following news of irregularities relating to Wirecard AG's financial statements and the alleged involvement of Singapore-based individuals and entities.",
"Area of Concern": ["Terrorism", "Cybersecurity"],
"Relevance Score": "High",
"Relationships": [
    ["Monetary Authority of Singapore", "Fined", ["DBS", "OCBC", "Citibank", "Swiss Life"]],
    ["DBS", "Fined", "Monetary Authority of Singapore"],
    ["OCBC", "Fined", "Monetary Authority of Singa

In [14]:
df = pd.read_csv("../data/news.csv")
df['terrorism'] = df["Text"].str.contains("terrorism", case=False)
df['security'] = df["Text"].str.contains("security", case=False)
df['espionage'] = df["Text"].str.contains("espionage", case=False)
df['communalism'] = df["Text"].str.contains("communalism|racial",regex=True, case=False)
df.to_csv("../data/news.csv", index=False)

In [17]:
df2 = pd.read_csv("../data/leaks.csv")
df2['terrorism'] = df2["Text"].str.contains("terrorism", case=False)
df2['security'] = df2["Text"].str.contains("security", case=False)
df2['espionage'] = df2["Text"].str.contains("espionage", case=False)
df2['communalism'] = df2["Text"].str.contains("communalism|racial",regex=True, case=False)
df2.to_csv("../data/leaks.csv", index=False)

In [22]:
df2['communalism'].describe()

count        44
unique        1
top       False
freq         44
Name: communalism, dtype: object